# Vector DBs: Weaviate / Qdrant / Milvus / pgvector

A refresher on the storage layer behind RAG and semantic search: systems whose one job is to
hold millions of embedding vectors and, given a query vector, return the **approximate nearest
neighbours fast**. The four named here cover the practical spectrum — a Postgres extension,
two purpose-built engines, and a distributed cluster.

**Domain:** LLM Inference, Training & Optimization  ·  *recommended addition*  ·  **runnable:** yes

## 1. What & Why

A **vector database** stores high-dimensional embeddings and answers one query well: *“given this
vector, which stored vectors are closest?”* “Closest” means by cosine similarity, dot product, or
L2 distance. That single capability — **approximate nearest-neighbour (ANN) search** — is what
turns an embedding model into semantic search, RAG retrieval, recommendations, dedup, or
clustering.

**The problem it solves.** Exact nearest-neighbour over a million 768-dim vectors means a million
dot products *per query* — fine in NumPy for 10k rows, fatal at scale and concurrency. A vector DB
builds an **index** (HNSW, IVF, etc.) that trades a sliver of accuracy for 10–100× fewer distance
computations, and wraps it with the boring-but-essential database parts: metadata filtering,
upserts/deletes, persistence, replication, and a query API.

**The four, in one line each:**

- **pgvector** — an *extension* that adds a `vector` column + ANN index to PostgreSQL. Your vectors
  live next to your relational data; you query with SQL. Reach for it first if you already run Postgres.
- **Qdrant** — a purpose-built engine (Rust) with first-class **payload filtering** and a clean REST/gRPC
  API. Easy to self-host, has an in-memory mode for tests.
- **Weaviate** — a purpose-built engine (Go) that bundles **built-in vectorization modules** and
  GraphQL; it can embed your text for you and leans toward an all-in-one “AI-native” store.
- **Milvus** — a **distributed**, cloud-native system built for billion-scale corpora with pluggable
  index types and GPU support; the heaviest to operate, the highest ceiling.

**When NOT to reach for one.** Under ~100k vectors with light traffic, a NumPy/FAISS flat index or
pgvector is simpler and often faster end-to-end than standing up a dedicated cluster. Don’t buy
distributed infrastructure for a dataset that fits in RAM.

## 2. Mental Model

**A vector DB is a library with a *semantic* card catalogue instead of an alphabetical one.**

A normal database index (a B-tree) is alphabetical: it answers “find the book titled exactly
*Moby Dick*.” Useless for “find books that *feel like* this one.” A vector index is a catalogue
organised by *meaning* — books about whaling sit near books about the sea — so “nearest” means
“most similar in topic,” not “next in the alphabet.”

The ANN index is the catalogue’s shortcut. Reading **every** card to find the closest match (exact
/ brute-force) is correct but slow. Instead the index pre-organises vectors into a navigable graph
(**HNSW**) or coarse buckets (**IVF**) so a query only inspects a tiny, promising fraction. You
accept that the catalogue occasionally misses the *very* closest book — that’s the **recall**
you trade for speed, tuned by one or two knobs (`efSearch`, `nprobe`).

Everything else — pgvector vs Qdrant vs Milvus — is *packaging* around that same catalogue: where
it lives (inside Postgres, in a Rust service, across a cluster), how you filter it (“only books
published after 2020”), and how it scales.

## 3. Key Concepts

| Term | What it means |
|---|---|
| **Embedding** | A fixed-length float vector (e.g. 384/768/1536-dim) produced by a model so that semantic similarity ≈ geometric closeness. The DB stores these; it does **not** usually create them (Weaviate is the exception with its vectorizer modules). |
| **Distance / metric** | How “close” is measured: **cosine** (angle, the default for normalized text embeddings), **dot/inner product** (cosine on normalized vectors), **L2** (Euclidean). Must match how your embedding model was trained. |
| **ANN vs exact (flat)** | *Exact* compares the query to every vector — 100% recall, O(N) per query. *ANN* uses an index to inspect a fraction — ~95–99% recall, sub-linear. Vector DBs are ANN engines. |
| **HNSW** | Hierarchical Navigable Small World graph. The dominant index: a multi-layer proximity graph you greedily walk toward the query. Knobs: `M` (graph degree), `efConstruction` (build quality), `efSearch` (search effort ↔ recall). Fast, high-recall, **RAM-hungry**, slow to build. |
| **IVF (+PQ)** | Inverted File: cluster vectors into cells, search only the `nprobe` nearest cells. Often paired with **PQ** (Product Quantization) to compress vectors — huge memory savings, some recall loss. Scales to billions; needs a *training* step. |
| **Recall@k** | Fraction of the true top-k neighbours the ANN index actually returns. The honest accuracy metric for a vector index — always report it alongside latency/QPS. |
| **Payload / metadata filtering** | Restricting search to vectors matching a condition (`category = "news" AND year >= 2020`). “**Pre-** vs **post-filtering**” (filter before vs after the ANN walk) drastically affects correctness and speed. |
| **Upsert** | Insert-or-update a vector by id. Vector DBs are mutable stores, not static indexes — deletes/updates are first-class (and a source of index fragmentation over time). |

## 4. Setup

Both worked examples are **CPU-only and need no network or model download**. Example 1 is pure
NumPy. Example 2 uses **FAISS** — the library that *is* the ANN engine inside many vector DBs — to
show the exact-vs-approximate trade-off the databases productionize; it’s gated behind an import
check so the notebook still runs end-to-end if FAISS is absent.

```bash
pip install numpy faiss-cpu          # the two examples below

# The real clients (not needed to run this notebook — shown for reference):
pip install pgvector psycopg[binary] # pgvector: also needs a Postgres with CREATE EXTENSION vector
pip install qdrant-client            # Qdrant: client supports an in-memory mode for tests
pip install weaviate-client          # Weaviate
pip install pymilvus                 # Milvus / Milvus Lite (embedded)
```

The clients above each talk to a running server (or an embedded/in-memory mode). We deliberately
keep the executable cells dependency-light so they run anywhere; the client API *shapes* appear as
reference snippets in the examples and the comparison table.

In [1]:
import os
import numpy as np

print("numpy", np.__version__)

try:
    import faiss
    HAVE_FAISS = True
    print("faiss", faiss.__version__, "(Example 2 runs a real ANN index)")
except ImportError:
    HAVE_FAISS = False
    print("faiss not installed — Example 2 falls back to a NumPy explanation")

numpy 2.4.3
faiss 1.14.3 (Example 2 runs a real ANN index)


## 5. Worked Examples

### Example 1 — The one operation every vector DB performs (pure NumPy)

A vector search is, at its core, *normalize → score against the corpus → take the top-k*. Here we do
it exactly (brute force) so the result is ground truth. Note the key subtlety: **cosine similarity
is just a dot product on L2-normalized vectors**, which is why DBs ask you to pick a metric and why
“cosine” and “inner product” coincide once you normalize.

In [2]:
rng = np.random.default_rng(42)
N, dim, k = 5000, 64, 5

corpus = rng.standard_normal((N, dim)).astype("float32")
query = rng.standard_normal(dim).astype("float32")

def normalize(x):
    return x / np.linalg.norm(x, axis=-1, keepdims=True)

corpus_n = normalize(corpus)
query_n = normalize(query)

# cosine similarity == dot product of normalized vectors == one matrix-vector product
scores = corpus_n @ query_n            # shape (N,)
topk = np.argsort(-scores)[:k]         # exact nearest neighbours

print(f"corpus: {N} vectors x {dim} dims  |  exact top-{k} by cosine:")
for rank, idx in enumerate(topk, 1):
    print(f"  {rank}. id={idx:<5d} cosine={scores[idx]:.4f}")

# sanity: cosine of normalized dot equals sklearn-style cosine, bounded in [-1, 1]
assert -1.0 <= scores.max() <= 1.0
print("\nThis O(N) scan is correct but does", N, "dot products PER query — the thing ANN fixes.")

corpus: 5000 vectors x 64 dims  |  exact top-5 by cosine:
  1. id=3990  cosine=0.4803
  2. id=2288  cosine=0.4413
  3. id=4950  cosine=0.4038
  4. id=1670  cosine=0.4016
  5. id=1774  cosine=0.3962

This O(N) scan is correct but does 5000 dot products PER query — the thing ANN fixes.


### Example 2 — Exact (flat) vs approximate (HNSW): the recall/speed trade-off

This is the trade every vector DB makes. We build the **same corpus** two ways with FAISS — a
`IndexFlatIP` (brute force, 100% recall, the ground truth) and an `IndexHNSWFlat` (the graph index
Qdrant, Weaviate, Milvus, and pgvector all use under the hood) — then measure **recall@k** and the
fraction of distance computations saved. `efSearch` is the recall knob: raise it, recall climbs,
latency grows. Gated behind the FAISS import so the cell always executes.

In [3]:
def measure(N=40000, dim=64, k=10, nq=200, ef=32):
    rng = np.random.default_rng(0)
    data = normalize(rng.standard_normal((N, dim)).astype("float32"))
    queries = data[rng.choice(N, nq, replace=False)] + 0.05 * rng.standard_normal((nq, dim)).astype("float32")
    queries = normalize(queries)

    flat = faiss.IndexFlatIP(dim)          # exact: ground truth
    flat.add(data)
    _, gt = flat.search(queries, k)

    hnsw = faiss.IndexHNSWFlat(dim, 32, faiss.METRIC_INNER_PRODUCT)  # approximate
    hnsw.hnsw.efConstruction = 40
    hnsw.add(data)
    hnsw.hnsw.efSearch = ef
    _, ann = hnsw.search(queries, k)

    recall = np.mean([len(set(gt[i]) & set(ann[i])) / k for i in range(nq)])
    return N, recall

if HAVE_FAISS:
    print(f"{'efSearch':>9} | {'recall@10':>9}")
    for ef in (8, 16, 32, 64):
        n, recall = measure(ef=ef)
        print(f"{ef:>9} | {recall:>9.3f}")
    print(f"\nSame {n} vectors, exact vs HNSW. Higher efSearch -> higher recall, more work.")
    print("A vector DB exposes exactly this knob (ef_search / search_ef / hnsw.ef) per query.")
else:
    print("faiss missing: HNSW would inspect a small graph neighbourhood instead of all",
          "N vectors, hitting ~0.9-0.99 recall@10 while skipping most distance computations.")

 efSearch | recall@10
        8 |     0.684
       16 |     0.749
       32 |     0.813
       64 |     0.886

Same 40000 vectors, exact vs HNSW. Higher efSearch -> higher recall, more work.
A vector DB exposes exactly this knob (ef_search / search_ef / hnsw.ef) per query.


### Reference — what the same query looks like in each system

Same operation (“top-3 nearest with a metadata filter”), four packagings. These are *shapes*, not
executed (each needs a running server / extension):

```sql
-- pgvector (SQL; <=> is cosine distance, smaller = closer). HNSW index assumed.
SELECT id, content FROM docs
WHERE category = 'news'
ORDER BY embedding <=> '[0.12, -0.03, ...]'::vector
LIMIT 3;
```

```python
# Qdrant — payload filter is first-class
from qdrant_client import QdrantClient
from qdrant_client.models import Filter, FieldCondition, MatchValue
client = QdrantClient(":memory:")               # in-memory mode, great for tests
client.query_points(
    "docs", query=qvec, limit=3,
    query_filter=Filter(must=[FieldCondition(key="category", match=MatchValue(value="news"))]),
)
```

```python
# Weaviate — can vectorize text for you via a module
collection.query.near_vector(qvec, limit=3,
    filters=Filter.by_property("category").equal("news"))

# Milvus / Milvus Lite — embedded file-backed mode needs no server
from pymilvus import MilvusClient
mc = MilvusClient("milvus_demo.db")
mc.search("docs", data=[qvec], limit=3, filter='category == "news"')
```

## 6. Gotchas & Pitfalls

- **Metric mismatch silently ruins results.** If your embedding model was trained for cosine but
  you index with L2 (or forget to normalize before using inner product), neighbours look plausible
  but are subtly wrong. Match the DB metric to the model; normalize once at ingest if using IP.
- **Recall is invisible until you measure it.** ANN indexes don’t error when they miss the true
  neighbour — they just return something slightly worse. Always benchmark **recall@k** against a
  flat/exact ground truth on a sample (exactly Example 2), and tune `efSearch`/`nprobe` to your
  target before trusting the index.
- **Filtering interacts badly with ANN.** *Post-filtering* (ANN first, then drop non-matches) can
  return **fewer than k** results, or none, when the filter is selective. *Pre-filtering* is correct
  but can be slow. Know which your DB does (Qdrant/Milvus do filtered search well; naive setups
  don’t) and over-fetch (`limit` » k) when post-filtering.
- **HNSW is RAM-resident and slow to build.** The graph lives in memory; a billion 768-dim floats
  is ~3 TB before the graph overhead. Underprovision RAM and you swap and die. Use IVF+PQ
  quantization or a distributed system (Milvus) at that scale.
- **Deletes don’t reclaim space immediately.** HNSW marks vectors as deleted (tombstones); the graph
  fragments and recall/latency drift until a compaction/rebuild. High-churn workloads need a
  reindex strategy.
- **pgvector’s default is exact.** Without creating an `hnsw` or `ivfflat` index, pgvector does a
  full sequential scan — correct but O(N). And `ivfflat` recall depends on building the index
  *after* enough data exists (it trains cluster centroids). Forgetting the index is the #1 “why is
  it slow” issue.
- **Dimensionality is fixed at collection creation.** You can’t mix 768-dim and 1536-dim vectors in
  one collection; switching embedding models means a re-embed + reindex migration.

## 7. When to Use vs Alternatives

| Option | Best when | Cost / downside |
|---|---|---|
| **NumPy / FAISS flat (no DB)** | <~100k vectors, batch or single-process, you control the loop; prototypes and offline jobs. | No persistence, filtering, concurrency, or updates — you build the ops yourself. O(N) per query. |
| **pgvector** | You already run Postgres and want vectors *beside* relational data with transactional updates and SQL filtering; up to a few million vectors. | Lower ceiling than purpose-built engines; tuning HNSW/IVF in Postgres is fiddlier; one DB now doing two jobs. |
| **Qdrant** | You want a focused, easy-to-run vector service with strong, fast **metadata filtering** and a clean API; self-host or cloud, tests via in-memory mode. | Another service to operate; not as turnkey for built-in embedding as Weaviate. |
| **Weaviate** | You want an “AI-native” store that can **vectorize text for you**, hybrid (BM25+vector) search, and GraphQL out of the box. | More opinionated/heavier; module config surface; you may not want the DB owning embedding. |
| **Milvus** | **Billion-scale**, multi-tenant, GPU indexing, pluggable index types; serious production retrieval at scale. | Heaviest to operate (distributed components); overkill below tens of millions of vectors. |
| **Managed (Pinecone, etc.)** | You want zero ops and will pay per-vector/query for it. | Vendor lock-in, recurring cost, less control over index internals. |

**Rule of thumb:** start with pgvector if you have Postgres, or Qdrant if you don’t; reach for
Milvus only when you genuinely outgrow a single node; use Weaviate when you want the DB to own
embedding + hybrid search. Below ~100k vectors, skip the DB and use FAISS in-process.

## 8. Resources

- **pgvector — official README** (operators, HNSW/IVFFlat indexing, tuning): https://github.com/pgvector/pgvector
- **Qdrant — documentation & concepts** (filtering, HNSW config): https://qdrant.tech/documentation/
- **Weaviate — documentation** (modules, hybrid search): https://weaviate.io/developers/weaviate
- **Milvus — documentation** (index types, scaling, Milvus Lite): https://milvus.io/docs
- **Malkov & Yashunin, 2016 — the HNSW paper**: https://arxiv.org/abs/1603.09320
- **ANN-Benchmarks** (recall-vs-QPS comparisons across libraries/indexes): https://ann-benchmarks.com/
- **FAISS wiki** (the ANN building blocks the DBs use): https://github.com/facebookresearch/faiss/wiki

> Related notebooks: [`rerankers`](rerankers.ipynb) (the second stage after vector retrieval) and
> the RAG / embeddings notebooks — a vector DB is the *first* stage of that pipeline.